# ARC-AGI-3 ForgeHypothesisAgent — Kaggle Competition Notebook (v4.0)

**Architecture:** BFS-First Hybrid Agent — three tiers, zero LLM calls
**Strategy:** Tier 1 Offline BFS solver → Tier 2 pure-Python rule-discovery → Tier 3 emergency heuristic
**Budget:** No model inference of any kind. 12-hour session limit is spent entirely on game-playing, not GPU inference.

This notebook implements **ARC_AGI3_Planning_Document_v4.md** in full. It retires the
Qwen2-VL-backed `HypothesisAgent` from v3.0 because even with a 2-5s/call local model,
2 calls × 100 actions × 55 games is 300-750 GPU-hours — impossible inside a 12-hour session.

Key differences from the v3.0 notebook:
- **No model downloads, no `autoawq`, no `qwen-vl-utils`, no GPU dependency at all.**
- `agents/templates/my_agent.py` is a **single file** containing `ForgeHypothesisAgent`,
  `BFSSolver`, and all the pure-Python utility logic (`RuleBook`, `StateGraph`, `FrameDelta`,
  `PhaseController`, BFS pathfinder, goal detector) — ported from the v3.0 `agents/utils/*`
  modules with every LLM call stripped out and replaced by deterministic rule update /
  action selection (Planning Doc Sections 2.2 and 2.3).
- All FORGE v19 bugs F1–F7 from Planning Doc Section 3.1 are fixed in this file.
- `agents/__init__.py` now maps `AVAILABLE_AGENTS` directly to the agent **class**
  (matching what `swarm.py`'s `AVAILABLE_AGENTS[agent]` actually expects) instead of to
  `None`, which was a latent bug in the v3.0 notebook that would have crashed `Swarm.main()`
  the moment it tried to instantiate an agent.

## STEP 1 — Copy ARC-AGI-3-Agents directory & wait for gateway

No wheel installation step is needed: v4 has zero extra dependencies beyond what the
competition environment already provides (`arc-agi`, `numpy`, `python-dotenv`). Section 9
of the planning document explicitly forbids any LLM/model-inference dependency.

In [1]:
import os
import shutil

src = '/kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents'
dst = '/kaggle/working/ARC-AGI-3-Agents'

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    if not os.path.exists(dst):
        print(f'Copying {src} -> {dst}')
        shutil.copytree(src, dst)
    else:
        print(f'Already exists: {dst}')
    print('Waiting for gateway...')
    os.system(
        'curl --fail --retry 999 --retry-all-errors --retry-delay 5 '
        '--retry-max-time 600 http://gateway:8001/api/games'
    )
else:
    if os.path.exists(src):
        if not os.path.exists(dst):
            print(f'Copying {src} -> {dst}')
            shutil.copytree(src, dst)
        else:
            print(f'Already exists: {dst}')
    else:
        os.makedirs(dst, exist_ok=True)

# Single-file structure (Planning Doc Section 7) — only templates/ is needed,
# no agents/utils/ package this time.
os.makedirs(f'{dst}/agents/templates', exist_ok=True)
print('Setup complete.')

Copying /kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents -> /kaggle/working/ARC-AGI-3-Agents
Setup complete.


## STEP 2 — Write `agents/templates/my_agent.py`

This is the entire v4 agent in one file (Planning Doc Section 7: *"Why single-file? FORGE
v19 used a single `my_agent.py` and it worked... For competition submission, single file
with inline classes is the safest structure."*). It contains, in order:

1. `FrameDelta` / `compute_delta` / `get_full_hash` / `get_abstract_hash` / `classify_decorative_values` (ported from `frame_diff.py`, Section 2.3 + 2.11)
2. `RuleBook` / `ActionEffect` / `ObjectDef` (ported from `rule_book.py`, Section 2.4)
3. `detect_goal_candidates` (ported from `goal_detector.py`, Section 2.6 — **Section 3.3 import-order fix applied**: `Optional` is imported once at the top of the file, not re-imported inside the function)
4. `bfs_path` / `find_nearest_value` (ported from `pathfinder.py`, Section 2.7)
5. `StateGraph` / `StateNode` (ported from `state_graph.py`, Section 2.5 + 2.11)
6. `PhaseController` / `Phase` (ported from `phase_controller.py` — `determine_reasoning_effort` dropped per Section 1.3, since there is no LLM call to budget effort for)
7. `BFSSolver` (FORGE v19, with bug fixes **F1, F3, F5, F7** applied — see Section 3.1)
8. `ForgeHypothesisAgent` — the three-tier agent (Section 5)

**Bug fixes applied (Planning Doc Section 3.1):**

| # | Bug | Fix |
|---|---|---|
| F1 | `bfs_timeout` referenced but never defined in `_try_bfs_solve` — `NameError` at runtime | Computed explicitly from the per-level time budget and passed as `timeout=` |
| F2 | `s.total_time_budget` etc. assigned at class-body scope with undefined `s` — `AttributeError` waiting to happen, time-budget logic dead | Moved into `__init__`, wired into `_on_level_change` (Section 4.4) |
| F3 | `_probe_hidden_fields` skipped all private fields except two — missed most hidden game state | `_serialize_game_state` now probes every comparable attribute (scalars, strings, tuples, lists, ndarrays), excluding only a short list of engine-internal fields |
| F4 | CNN revisit reward `+0.2` rewarded loops | N/A in v4 — the CNN fallback itself is retired; Tier 2 has no equivalent reward-shaping bug |
| F5 | BFS depth cap hard-coded at 30 | `adaptive_depth()` picks 25–60 based on action-space size and scan time (Section 4.1) |
| F6 | AEM tensors built but never passed into CNN training | N/A in v4 — no CNN, no AEM |
| F7 | `find_game_source_and_class` glob patterns fragile, no warning logged | Added explicit `/kaggle/input/competitions/.../**/{gid}.py` fallback pattern and logs every path attempted when source isn't found |

Also fixes the **Section 3.3** `goal_detector.py` issue (the `from typing import Optional`
import was duplicated mid-file in the old notebook) by importing `Optional` once at the top.

In [2]:
%%writefile /kaggle/working/ARC-AGI-3-Agents/agents/templates/my_agent.py
# =====================================================================
# ForgeHypothesisAgent — v4.0 BFS-First Hybrid Agent
#
# Implements ARC_AGI3_Planning_Document_v4.md in full:
#   Tier 1 — Offline BFS Solver (FORGE v19, bug-fixed F1-F7)
#   Tier 2 — Pure-Python Rule-Discovery (arc_prize_2026 utils, LLM stripped)
#   Tier 3 — Emergency Heuristic (FORGE v19 _heuristic())
#
# No LLM. No model inference. No GPU required. Single file (Section 7).
#
# Bug fixes applied (Section 3.1):
#   F1 - bfs_timeout NameError in _try_bfs_solve -> pass explicit timeout
#   F2 - class-body `s.` assignments (undefined `s`) -> moved into __init__
#   F3 - _probe_hidden_fields skipped most private fields -> probe all but
#        a short list of internal engine fields
#   F4 - revisit reward was +0.2 (rewards loops) -> -0.3 (kept for reference;
#        the CNN fallback itself is retired in v4, logic kept dormant/unused)
#   F5 - BFS depth cap hard-coded 30 -> adaptive (Section 4.1), default 45/35
#   F6 - AEM tensors never used in training -> N/A, CNN fallback retired
#   F7 - find_game_source_and_class fragile globs, no warning -> added
#        explicit Kaggle competition path + logging of every path attempted
#
# Section 3.3 fix: goal_detector Optional import moved to top of file.
# =====================================================================
import copy
import glob
import hashlib
import importlib.util
import logging
import os
import random
import re
import time
import traceback
from collections import Counter, deque
from typing import Any, Dict, List, Optional, Set, Tuple

import numpy as np

from agents.agent import Agent
from arcengine import FrameData, GameAction, GameState, ActionInput

logger = logging.getLogger(__name__)


# =====================================================================
# SECTION: frame_diff.py (ported, pure Python, no LLM)
# =====================================================================

class FrameDelta:
    """Structured diff between two consecutive frames (Section 2.3)."""

    __slots__ = (
        'cells_changed', 'changed_cells', 'player_delta', 'level_changed',
        'state_changed', 'new_values_seen', 'grid_hash_before', 'grid_hash_after',
        'abstract_hash_before', 'abstract_hash_after',
    )

    def __init__(self, cells_changed, changed_cells, player_delta, level_changed,
                 state_changed, new_values_seen, grid_hash_before, grid_hash_after,
                 abstract_hash_before, abstract_hash_after):
        self.cells_changed = cells_changed
        self.changed_cells = changed_cells
        self.player_delta = player_delta
        self.level_changed = level_changed
        self.state_changed = state_changed
        self.new_values_seen = new_values_seen
        self.grid_hash_before = grid_hash_before
        self.grid_hash_after = grid_hash_after
        self.abstract_hash_before = abstract_hash_before
        self.abstract_hash_after = abstract_hash_after


def get_full_hash(grid) -> str:
    """Exact hash for transition_cache (Section 2.10)."""
    if isinstance(grid, np.ndarray):
        return hashlib.sha1(grid.tobytes()).hexdigest()
    return hashlib.sha1(str(grid).encode()).hexdigest()


def get_abstract_hash(grid, decorative_values: Set[int]) -> str:
    """Hash with decorative values zeroed out — used for StateGraph (Section 2.11)."""
    if not decorative_values:
        return get_full_hash(grid)
    masked = [[0 if v in decorative_values else v for v in row] for row in grid]
    return hashlib.sha1(str(masked).encode()).hexdigest()


def _detect_player_delta(changed_cells, before, after) -> Optional[Tuple[int, int]]:
    """Infer player movement from coherent cluster shift in changed cells."""
    if not changed_cells or len(changed_cells) > 50:
        return None
    left = [(r, c, ov) for r, c, ov, nv in changed_cells if ov != 0 and nv == 0]
    arrived = [(r, c, nv) for r, c, ov, nv in changed_cells if ov == 0 and nv != 0]
    if not left or not arrived or len(left) != len(arrived):
        return None
    dr_set, dc_set = set(), set()
    for (r1, c1, _), (r2, c2, _) in zip(sorted(left), sorted(arrived)):
        dr_set.add(r2 - r1)
        dc_set.add(c2 - c1)
    if len(dr_set) == 1 and len(dc_set) == 1:
        return (dr_set.pop(), dc_set.pop())
    return None


def compute_delta(
    grid_before, grid_after, levels_before, levels_after,
    state_before, state_after, known_values, decorative_values,
) -> FrameDelta:
    """Compute structured FrameDelta between two 64x64 grids."""
    before = np.array(grid_before, dtype=np.int16)
    after = np.array(grid_after, dtype=np.int16)

    diff_mask = before != after
    changed_indices = np.argwhere(diff_mask)
    changed_cells = [
        (int(r), int(c), int(before[r, c]), int(after[r, c]))
        for r, c in changed_indices
    ]

    after_values = set(int(v) for v in np.unique(after))
    new_values_seen = after_values - known_values
    player_delta = _detect_player_delta(changed_cells, before, after)

    full_before = get_full_hash(grid_before)
    full_after = get_full_hash(grid_after)
    abs_before = get_abstract_hash(grid_before, decorative_values)
    abs_after = get_abstract_hash(grid_after, decorative_values)

    return FrameDelta(
        cells_changed=int(diff_mask.sum()),
        changed_cells=changed_cells,
        player_delta=player_delta,
        level_changed=(levels_after != levels_before),
        state_changed=(state_before != state_after),
        new_values_seen=new_values_seen,
        grid_hash_before=full_before,
        grid_hash_after=full_after,
        abstract_hash_before=abs_before,
        abstract_hash_after=abs_after,
    )


def classify_decorative_values(
    grid_history, player_deltas, level_changed_flags, state_changed_flags,
    goal_candidate_values,
) -> Set[int]:
    """Identify values that move on their own (enemies, particles, animations)."""
    if len(grid_history) < 3:
        return set()
    decorative: Set[int] = set()
    n = len(grid_history)
    all_values: Set[int] = set()
    for g in grid_history:
        for row in g:
            all_values.update(row)

    for val in all_values:
        if val == 0 or val in goal_candidate_values:
            continue
        move_count = 0
        for i in range(1, n):
            if level_changed_flags[i] or state_changed_flags[i]:
                continue
            g_prev = np.array(grid_history[i - 1])
            g_curr = np.array(grid_history[i])
            prev_pos = set(map(tuple, np.argwhere(g_prev == val).tolist()))
            curr_pos = set(map(tuple, np.argwhere(g_curr == val).tolist()))
            if prev_pos != curr_pos and len(prev_pos) > 0:
                pd = player_deltas[i] if i < len(player_deltas) else None
                if pd is None:
                    move_count += 1
                else:
                    offset_match = any(
                        (r + pd[0], c + pd[1]) in curr_pos for r, c in prev_pos
                    )
                    if not offset_match:
                        move_count += 1
        if move_count >= 2:
            decorative.add(val)
    return decorative


# =====================================================================
# SECTION: rule_book.py (ported, pure Python, no LLM)
# =====================================================================

class ActionEffect:
    """What we know about a single action's effect."""

    def __init__(self, action_name, direction=None, avg_cells_changed=0.0,
                 effect_type='unknown', requires_coordinates=False, observation_count=0):
        self.action_name = action_name
        self.direction = direction
        self.avg_cells_changed = avg_cells_changed
        self.effect_type = effect_type
        self.requires_coordinates = requires_coordinates
        self.observation_count = observation_count


class ObjectDef:
    def __init__(self, name, values, behavior='', locations=None):
        self.name = name
        self.values = values
        self.behavior = behavior
        self.locations = locations or []


class RuleBook:
    """Persistent in-session knowledge accumulator (Section 2.4)."""

    def __init__(self) -> None:
        self.action_map: Dict[str, ActionEffect] = {}
        self.no_op_actions: List[str] = []

        self.impassable_values: List[int] = []
        self.passable_values: List[int] = []
        self.player_position: Optional[Tuple[int, int]] = None

        self.objects: Dict[str, ObjectDef] = {}

        self.win_condition: Optional[str] = None
        self.current_objective: str = 'Explore the environment and discover game rules.'
        self.goal_candidates: List[Tuple[int, float]] = []

        self.confirmed_rules: List[str] = []
        self.tentative_rules: List[str] = []
        self.retracted_rules: List[str] = []

        self.levels_completed: int = 0
        self.action_budget_used: int = 0
        self.available_actions: List[str] = []
        self.known_values: Set[int] = set()

        # Efficiency layer
        self.transition_cache: Dict[Tuple[str, str], str] = {}
        self.decorative_values: Set[int] = set()
        self.cached_bfs_path: Optional[List[str]] = None
        self.cached_bfs_goal: Optional[Tuple[int, int]] = None

    def carry_forward_for_new_level(self) -> None:
        """On level transition: keep movement/wall rules, reset goal model."""
        movement_rules = [r for r in self.confirmed_rules if any(
            kw in r.lower() for kw in ['move', 'wall', 'floor', 'pass', 'impass']
        )]
        self.win_condition = None
        self.current_objective = 'New level: re-identify the goal and win condition.'
        self.goal_candidates = []
        self.cached_bfs_path = None
        self.cached_bfs_goal = None
        self.confirmed_rules = movement_rules
        self.tentative_rules = []
        # transition_cache and decorative_values persist (Section 2.10 lifecycle)


# =====================================================================
# SECTION: goal_detector.py (ported, Section 3.3 import fix applied —
# Optional imported at top of file, not inside the function)
# =====================================================================

def detect_goal_candidates(
    grid_history: List[List[List[int]]],
    impassable_values: List[int],
    floor_values: Optional[List[int]] = None,
    weights: Tuple[float, float, float] = (0.5, 0.3, 0.2),
) -> List[Tuple[int, float]]:
    """Return [(value, goal_candidate_score), ...] sorted descending."""
    if floor_values is None:
        floor_values = [0]
    if not grid_history:
        return []

    latest = grid_history[-1]
    rows, cols = len(latest), len(latest[0])
    grid_np = np.array(latest, dtype=np.int16)

    unique, counts = np.unique(grid_np, return_counts=True)
    excluded = set(impassable_values) | set(floor_values) | {0}

    w_rarity, w_edge, w_change = weights
    scores: Dict[int, float] = {}

    for value, count in zip(unique.tolist(), counts.tolist()):
        if value in excluded:
            continue
        rarity_weight = 1.0 - min(count / 10.0, 1.0)

        positions = np.argwhere(grid_np == value)
        on_edge = sum(
            1 for r, c in positions
            if r < 2 or r >= rows - 2 or c < 2 or c >= cols - 2
        )
        edge_weight = on_edge / max(len(positions), 1)

        change_count = 0
        for i in range(1, len(grid_history)):
            prev = np.array(grid_history[i - 1], dtype=np.int16)
            curr = np.array(grid_history[i], dtype=np.int16)
            if prev.shape == curr.shape:
                changed = (prev != curr) & ((prev == value) | (curr == value))
                if changed.any():
                    change_count += 1
        change_weight = min(change_count / max(len(grid_history) - 1, 1), 1.0)

        scores[int(value)] = (
            w_rarity * rarity_weight + w_edge * edge_weight + w_change * change_weight
        )

    return sorted(scores.items(), key=lambda kv: kv[1], reverse=True)


# =====================================================================
# SECTION: pathfinder.py (ported, pure Python BFS)
# =====================================================================

_MOVEMENT_ACTIONS = [
    (-1, 0, 'ACTION1'),   # Up
    (1, 0, 'ACTION2'),    # Down
    (0, -1, 'ACTION3'),   # Left
    (0, 1, 'ACTION4'),    # Right
]


def bfs_path(grid, start, target_value, impassable_values) -> Optional[List[str]]:
    """Shortest path from start to nearest cell with target_value."""
    rows, cols = len(grid), len(grid[0])
    start_r, start_c = start
    if grid[start_r][start_c] == target_value:
        return []
    impassable_set: Set[int] = set(impassable_values)
    queue = deque([(start, [])])
    visited: Set[Tuple[int, int]] = {start}
    while queue:
        (r, c), path = queue.popleft()
        for dr, dc, action in _MOVEMENT_ACTIONS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and (nr, nc) not in visited:
                cell_val = grid[nr][nc]
                if cell_val == target_value:
                    return path + [action]
                if cell_val not in impassable_set:
                    visited.add((nr, nc))
                    queue.append(((nr, nc), path + [action]))
    return None


def find_nearest_value(grid, start, target_value) -> Optional[Tuple[int, int]]:
    """BFS to find position of nearest cell with target_value (ignores impassable)."""
    rows, cols = len(grid), len(grid[0])
    queue = deque([start])
    visited: Set[Tuple[int, int]] = {start}
    while queue:
        r, c = queue.popleft()
        if grid[r][c] == target_value:
            return (r, c)
        for dr, dc, _ in _MOVEMENT_ACTIONS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and (nr, nc) not in visited:
                visited.add((nr, nc))
                queue.append((nr, nc))
    return None


# =====================================================================
# SECTION: state_graph.py (ported, pure Python)
# =====================================================================

class StateNode:
    def __init__(self, abstract_hash, parent_hash=None, action_from_parent=None):
        self.abstract_hash = abstract_hash
        self.parent_hash = parent_hash
        self.action_from_parent = action_from_parent
        self.tried_actions: Set[str] = set()
        self.visit_count = 0


class StateGraph:
    """Directed graph of visited abstract game states and transitions tried."""

    def __init__(self) -> None:
        self.states: Dict[str, StateNode] = {}

    @property
    def size(self) -> int:
        return len(self.states)

    def add_state(self, abstract_hash, action_taken=None, parent_hash=None) -> StateNode:
        if abstract_hash not in self.states:
            self.states[abstract_hash] = StateNode(abstract_hash, parent_hash, action_taken)
        node = self.states[abstract_hash]
        node.visit_count += 1
        if action_taken and parent_hash and parent_hash in self.states:
            self.states[parent_hash].tried_actions.add(action_taken)
        return node

    def get_untried_actions(self, abstract_hash, available_actions) -> List[str]:
        if abstract_hash not in self.states:
            return list(available_actions)
        tried = self.states[abstract_hash].tried_actions
        return [a for a in available_actions if a not in tried]

    def is_dead_end(self, abstract_hash, available_actions) -> bool:
        return len(self.get_untried_actions(abstract_hash, available_actions)) == 0

    def suggest_backtrack(self, current_hash, available_actions) -> List[str]:
        visited: Set[str] = {current_hash}
        queue = deque([[current_hash]])
        while queue:
            path = queue.popleft()
            node_hash = path[-1]
            if node_hash != current_hash:
                if not self.is_dead_end(node_hash, available_actions):
                    return self._path_to_actions(current_hash, node_hash)
            node = self.states.get(node_hash)
            if node and node.parent_hash and node.parent_hash not in visited:
                visited.add(node.parent_hash)
                queue.append(path + [node.parent_hash])
        return []

    def _path_to_actions(self, from_hash, to_hash) -> List[str]:
        path: List[str] = []
        current = from_hash
        visited: Set[str] = set()
        while current and current != to_hash and current not in visited:
            visited.add(current)
            node = self.states.get(current)
            if node and node.action_from_parent:
                path.append(node.action_from_parent)
                current = node.parent_hash or ''
            else:
                break
        return list(reversed(path))

    def is_stuck(self, recent_hashes, window=5, threshold=3) -> bool:
        if len(recent_hashes) < window:
            return False
        counts = Counter(recent_hashes[-window:])
        return counts.most_common(1)[0][1] >= threshold


# =====================================================================
# SECTION: phase_controller.py (ported — determine_reasoning_effort dropped,
# Section 1.3: no LLM means no adaptive-effort policy needed)
# =====================================================================

class Phase:
    EXPLORE = 'explore'
    VALIDATE = 'validate'
    EXPLOIT = 'exploit'
    RECOVERY = 'recovery'


class PhaseController:
    """Determines current game phase (Section 2.8). No LLM effort policy (2.9 dropped)."""

    @staticmethod
    def determine_phase(rule_book: RuleBook, action_counter: int,
                         state_graph: StateGraph, current_abstract_hash: str) -> str:
        available = rule_book.available_actions
        n_available = len(available) if available else 4

        if action_counter < n_available + 1:
            return Phase.EXPLORE
        if rule_book.win_condition is None:
            return Phase.VALIDATE
        if len(rule_book.confirmed_rules) < 3:
            return Phase.VALIDATE
        untried = state_graph.get_untried_actions(current_abstract_hash, available)
        if untried and action_counter < 75:
            return Phase.VALIDATE
        if action_counter > 75:
            return Phase.EXPLOIT
        return Phase.EXPLOIT if rule_book.win_condition else Phase.VALIDATE

    @staticmethod
    def suggest_exploration_action(rule_book: RuleBook) -> Optional[str]:
        tried = set(rule_book.action_map.keys())
        for action in rule_book.available_actions:
            if action not in tried:
                return action
        return None


# =====================================================================
# SECTION: action/direction maps shared by Tier 1 and Tier 2
# =====================================================================

_ACTION_DIRECTIONS = {
    'ACTION1': 'up', 'ACTION2': 'down', 'ACTION3': 'left', 'ACTION4': 'right',
    'ACTION5': 'interact', 'ACTION6': 'click', 'ACTION7': 'undo',
}
_DIRECTION_DELTAS = {
    'up': (-1, 0), 'down': (1, 0), 'left': (0, -1), 'right': (0, 1),
}
_ALL_ACTIONS = ['ACTION1', 'ACTION2', 'ACTION3', 'ACTION4', 'ACTION5', 'ACTION6', 'ACTION7']

# Engine-internal fields to always exclude from hidden-state probing (F3).
# Everything else (including private fields) is treated as candidate hidden state.
_ENGINE_INTERNAL_FIELDS = {'_action_count', '_full_reset', '_action_complete', '_frame_count'}


# =====================================================================
# SECTION: BFSSolver (Tier 1) — FORGE v19 base, bugs F1/F3/F5/F7 fixed
# =====================================================================

class BFSSolver:
    """Offline BFS solver using direct game class instantiation (Tier 1)."""

    def __init__(self, game_path, game_class_name, scan_timeout=5, bfs_timeout=120):
        self.game_path = game_path
        self.class_name = game_class_name
        self.scan_timeout = scan_timeout
        self.bfs_timeout = bfs_timeout
        self.game_cls = None
        self.solutions: Dict[int, list] = {}

    def load(self) -> bool:
        try:
            spec = importlib.util.spec_from_file_location('game_mod', self.game_path)
            mod = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(mod)
            self.game_cls = getattr(mod, self.class_name)
            return True
        except Exception as e:
            logger.warning(f"BFS: Failed to load game class: {e}")
            return False

    def _state_hash(self, g, frame, hidden_fields=None):
        fh = hashlib.md5(frame.tobytes()).hexdigest()[:16]
        if hidden_fields:
            extras = []
            for field_name in hidden_fields:
                try:
                    v = getattr(g, field_name, None)
                    if v is not None:
                        extras.append(f"{field_name}={v}")
                except Exception:
                    pass
            if extras:
                return fh + "|" + "|".join(extras)
        return fh

    def _serialize_game_state(self, game) -> dict:
        """F3/Section 4.3 — serialize ALL comparable attributes, not just scalars."""
        result = {}
        for k, v in game.__dict__.items():
            try:
                if isinstance(v, (int, float, bool, str, tuple, list)):
                    result[k] = v
                elif isinstance(v, np.ndarray):
                    result[k] = v.tobytes()
            except Exception:
                pass
        return result

    def _probe_hidden_fields(self, game, actions) -> List[str]:
        """F3 fix: probe ALL non-engine-internal fields, including private ones
        (e.g. _inventory, _door_open, _collected), not just two whitelisted names.
        Also covers list/tuple/str/ndarray fields, not only scalars (Section 4.3)."""
        if not actions:
            return []
        initial = self._serialize_game_state(game)
        changing_fields: Set[str] = set()

        for act_id, data in actions[:15]:
            g = copy.deepcopy(game)
            try:
                ai = ActionInput(id=GameAction.from_id(act_id), data=data) if data else ActionInput(id=GameAction.from_id(act_id))
                g.perform_action(ai, raw=True)
            except Exception:
                continue
            current = self._serialize_game_state(g)
            for k, v in current.items():
                if k in _ENGINE_INTERNAL_FIELDS:
                    continue
                if k in initial and initial[k] != v:
                    changing_fields.add(k)

        return sorted(changing_fields)

    def _scan_actions(self, game, f0, bg):
        avail = game._available_actions
        actions = []
        for a in [a for a in avail if a <= 5]:
            g = copy.deepcopy(game)
            try:
                r = g.perform_action(ActionInput(id=GameAction.from_id(a)), raw=True)
                if r.frame and np.sum(f0 != np.array(r.frame[-1])) > 0:
                    actions.append((a, None))
            except Exception:
                pass
        if 6 in avail:
            t0 = time.time()
            seen_effects = set()
            for y in range(0, 64, 2):
                if time.time() - t0 > self.scan_timeout:
                    break
                for x in range(0, 64, 2):
                    if f0[y, x] == bg:
                        continue
                    g = copy.deepcopy(game)
                    try:
                        r = g.perform_action(
                            ActionInput(id=GameAction.ACTION6, data={'x': x, 'y': y, 'game_id': 'bfs'}),
                            raw=True
                        )
                        if not r.frame:
                            continue
                        f = np.array(r.frame[-1])
                        diff = np.sum(f0 != f)
                        if diff > 0:
                            effect_hash = hashlib.md5(f.tobytes()).hexdigest()[:12]
                            if effect_hash not in seen_effects:
                                seen_effects.add(effect_hash)
                                actions.append((6, {'x': x, 'y': y, 'game_id': 'bfs'}))
                    except Exception:
                        pass
        return actions

    @staticmethod
    def adaptive_depth(actions: list, scan_time: float) -> int:
        """F5 fix (Section 4.1): replace hard-coded depth=30 with adaptive depth."""
        n_actions = len(actions)
        if n_actions <= 2:
            return 60
        elif n_actions <= 4:
            return 45
        elif scan_time > 2.0:
            return 25
        else:
            return 35

    def solve_level(self, level_idx, max_states=500000, prev_solution=None, timeout=None):
        """Find optimal solution for a level via BFS (memory-optimised replay)."""
        if not self.game_cls:
            return None

        effective_timeout = timeout if timeout is not None else self.bfs_timeout

        game = self.game_cls()
        game.set_level(level_idx)
        game.perform_action(ActionInput(id=GameAction.RESET), raw=True)

        r0 = game.perform_action(ActionInput(id=GameAction.RESET), raw=True)
        if not r0.frame:
            return None
        f0 = np.array(r0.frame[-1])
        bg = int(np.bincount(f0.flatten(), minlength=16).argmax())

        if prev_solution and level_idx > 0:
            transfer_result = self._try_transfer(game, level_idx, prev_solution, f0)
            if transfer_result:
                return transfer_result

        scan_t0 = time.time()
        actions = self._scan_actions(game, f0, bg)
        scan_time = time.time() - scan_t0

        if not actions:
            avail = game._available_actions
            for warmup_id in [a for a in avail if a <= 4]:
                g_warmup = copy.deepcopy(game)
                try:
                    g_warmup.perform_action(ActionInput(id=GameAction.from_id(warmup_id)), raw=True)
                    f_after = np.array(g_warmup.get_pixels(0, 0, 64, 64))
                    warmup_actions = self._scan_actions(g_warmup, f_after, bg)
                    if warmup_actions:
                        logger.info(f"BFS L{level_idx}: UNLOCKED with ACTION{warmup_id}! {len(warmup_actions)} actions")
                        game = g_warmup
                        f0 = f_after
                        actions = warmup_actions
                        break
                except Exception:
                    pass

        logger.info(f"BFS L{level_idx}: {len(actions)} effective actions")
        if not actions:
            return None

        # F5: adaptive depth cap instead of hard-coded 30
        max_depth = self.adaptive_depth(actions, scan_time)

        hidden_fields = None
        visited = set()
        queue = deque()
        h0 = self._state_hash(game, f0, None)
        visited.add(h0)

        base_game = copy.deepcopy(game)
        queue.append(([], 0))

        t0 = time.time()
        explored = 0

        while queue and explored < max_states and (time.time() - t0) < effective_timeout:
            hist, depth = queue.popleft()

            g = copy.deepcopy(base_game)
            try:
                for a_id, a_data in hist:
                    ai = ActionInput(id=GameAction.from_id(a_id), data=a_data) if a_data else ActionInput(id=GameAction.from_id(a_id))
                    g.perform_action(ai, raw=True)
            except Exception:
                continue

            for act_id, data in actions:
                g2 = copy.deepcopy(g)
                try:
                    ai = ActionInput(id=GameAction.from_id(act_id), data=data) if data else ActionInput(id=GameAction.from_id(act_id))
                    r = g2.perform_action(ai, raw=True)
                except Exception:
                    continue
                explored += 1

                if not r.frame:
                    continue
                f = np.array(r.frame[-1])
                h = self._state_hash(g2, f, hidden_fields)
                if h in visited:
                    continue
                visited.add(h)

                new_hist = hist + [(act_id, data)]

                if r.levels_completed > level_idx or g2._current_level_index > level_idx:
                    elapsed = time.time() - t0
                    logger.info(f"BFS L{level_idx}: SOLVED in {len(new_hist)} actions ({explored} explored, {elapsed:.1f}s)")
                    self.solutions[level_idx] = new_hist
                    return new_hist

                if depth < max_depth:
                    queue.append((new_hist, depth + 1))

        elapsed_first = time.time() - t0
        logger.info(f"BFS L{level_idx}: first pass timeout ({explored} explored, {len(visited)} unique, {elapsed_first:.1f}s)")

        if explored < 20 and elapsed_first > 10.0:
            logger.info(f"BFS L{level_idx}: early exit (only {explored} explored in {elapsed_first:.1f}s) — handing off to Tier 2")
            return None

        if len(visited) < 50 and elapsed_first < effective_timeout * 0.8:
            # F3: full-field hidden-state probe (private fields included)
            hidden_fields = self._probe_hidden_fields(game, actions)
            if hidden_fields:
                logger.info(f"BFS L{level_idx}: RETRY with hidden fields: {hidden_fields}")

                # F1/Fix3 (kept): exactly 2 RESET calls to match the first-pass baseline
                game2 = self.game_cls()
                game2.set_level(level_idx)
                game2.perform_action(ActionInput(id=GameAction.RESET), raw=True)
                r0_2 = game2.perform_action(ActionInput(id=GameAction.RESET), raw=True)
                if not r0_2.frame:
                    return None
                f0_2 = np.array(r0_2.frame[-1])
                h0_2 = self._state_hash(game2, f0_2, hidden_fields)

                base_game2 = copy.deepcopy(game2)
                visited2 = set()
                visited2.add(h0_2)
                queue2 = deque()
                queue2.append(([], 0))

                t0_2 = time.time()
                explored2 = 0
                remaining = max(30, effective_timeout - elapsed_first)
                # F5: shallower depth cap for the hidden-state retry pass
                retry_max_depth = min(max_depth, 40)

                while queue2 and explored2 < max_states and (time.time() - t0_2) < remaining:
                    hist, depth = queue2.popleft()

                    g = copy.deepcopy(base_game2)
                    try:
                        for a_id, a_data in hist:
                            ai = ActionInput(id=GameAction.from_id(a_id), data=a_data) if a_data else ActionInput(id=GameAction.from_id(a_id))
                            g.perform_action(ai, raw=True)
                    except Exception:
                        continue

                    for act_id, data in actions:
                        g2 = copy.deepcopy(g)
                        try:
                            ai = ActionInput(id=GameAction.from_id(act_id), data=data) if data else ActionInput(id=GameAction.from_id(act_id))
                            r = g2.perform_action(ai, raw=True)
                        except Exception:
                            continue
                        explored2 += 1

                        if not r.frame:
                            continue
                        f = np.array(r.frame[-1])
                        h = self._state_hash(g2, f, hidden_fields)
                        if h in visited2:
                            continue
                        visited2.add(h)

                        new_hist = hist + [(act_id, data)]

                        if r.levels_completed > level_idx or g2._current_level_index > level_idx:
                            logger.info(f"BFS L{level_idx}: SOLVED (hidden retry) in {len(new_hist)} actions ({explored2} explored)")
                            self.solutions[level_idx] = new_hist
                            return new_hist

                        if depth < retry_max_depth:
                            queue2.append((new_hist, depth + 1))

                logger.info(f"BFS L{level_idx}: hidden retry also failed ({explored2} explored, {len(visited2)} unique)")

        return None

    def _try_transfer(self, game, level_idx, prev_solution, f1):
        """Transfer previous level's solution to current level."""
        try:
            g = copy.deepcopy(game)
            for i, (act_id, data) in enumerate(prev_solution):
                try:
                    ai = ActionInput(id=GameAction.from_id(act_id), data=data) if data else ActionInput(id=GameAction.from_id(act_id))
                    r = g.perform_action(ai, raw=True)
                    if r.levels_completed > level_idx or g._current_level_index > level_idx:
                        logger.info(f"BFS L{level_idx}: TRANSFER SUCCESS (direct replay, {i+1} actions)")
                        sol = prev_solution[:i + 1]
                        self.solutions[level_idx] = sol
                        return sol
                except Exception:
                    break

            prev_game = self.game_cls()
            prev_game.set_level(level_idx - 1)
            prev_game.perform_action(ActionInput(id=GameAction.RESET), raw=True)
            r_prev = prev_game.perform_action(ActionInput(id=GameAction.RESET), raw=True)
            if not r_prev.frame:
                return None
            f0 = np.array(r_prev.frame[-1])
            bg = int(np.bincount(f0.flatten(), minlength=16).argmax())

            def get_objects(frame, bg_c):
                objs = []
                for c in range(16):
                    if c == bg_c:
                        continue
                    mask = (frame == c)
                    npix = int(np.sum(mask))
                    if npix < 2:
                        continue
                    ys, xs = np.where(mask)
                    objs.append({'color': c, 'cx': float(np.mean(xs)), 'cy': float(np.mean(ys)), 'n': npix})
                return sorted(objs, key=lambda o: (o['color'], -o['n']))

            objs_prev = get_objects(f0, bg)
            objs_curr = get_objects(f1, bg)
            if not objs_prev or not objs_curr:
                return None

            matched = []
            for op in objs_prev:
                best, best_dist = None, float('inf')
                for oc in objs_curr:
                    if oc['color'] == op['color'] and abs(oc['n'] - op['n']) < max(op['n'], oc['n']) * 0.5:
                        d = abs(oc['cx'] - op['cx']) + abs(oc['cy'] - op['cy'])
                        if d < best_dist:
                            best_dist = d
                            best = oc
                if best:
                    matched.append((op, best))

            if not matched:
                return None

            dx = np.mean([m[1]['cx'] - m[0]['cx'] for m in matched])
            dy = np.mean([m[1]['cy'] - m[0]['cy'] for m in matched])

            transferred = []
            for act_id, data in prev_solution:
                if data and 'x' in data:
                    new_data = dict(data)
                    new_data['x'] = max(0, min(63, int(data['x'] + dx)))
                    new_data['y'] = max(0, min(63, int(data['y'] + dy)))
                    transferred.append((act_id, new_data))
                else:
                    transferred.append((act_id, data))

            g = copy.deepcopy(game)
            for i, (act_id, data) in enumerate(transferred):
                try:
                    ai = ActionInput(id=GameAction.from_id(act_id), data=data) if data else ActionInput(id=GameAction.from_id(act_id))
                    r = g.perform_action(ai, raw=True)
                    if r.levels_completed > level_idx or g._current_level_index > level_idx:
                        logger.info(f"BFS L{level_idx}: TRANSFER SUCCESS (offset dx={dx:.0f},dy={dy:.0f}, {i+1} actions)")
                        sol = transferred[:i + 1]
                        self.solutions[level_idx] = sol
                        return sol
                except Exception:
                    break

        except Exception as e:
            logger.warning(f"BFS transfer failed: {e}")
        return None


def find_game_source_and_class(game_id, arc_env=None):
    """F7 fix: explicit Kaggle competition fallback path + logs every path attempted."""
    gid = game_id.split('-')[0]
    cls_name = gid.capitalize()
    if len(gid) == 4 and gid[0].isalpha():
        cls_name = gid[0].upper() + gid[1:]

    attempted: List[str] = []
    src = None

    if arc_env and hasattr(arc_env, 'environment_info'):
        ei = arc_env.environment_info
        if hasattr(ei, 'local_dir') and ei.local_dir:
            from pathlib import Path
            ld = Path(ei.local_dir)
            for candidate in [ld / f"{gid}.py", ld / f"{cls_name.lower()}.py"]:
                attempted.append(str(candidate))
                if candidate.exists():
                    src = str(candidate)
                    content = candidate.read_text()[:2000]
                    m = re.search(r'class\s+(\w+)\s*\(\s*ARCBaseGame', content)
                    if m:
                        cls_name = m.group(1)
                    break

    if not src:
        patterns = [
            f"/tmp/*/{gid}/*/{gid}.py",
            f"/kaggle/*/{gid}*/{gid}.py",
            # F7: explicit competition data path fallback
            f"/kaggle/input/competitions/arc-prize-2026-arc-agi-3/**/{gid}.py",
            f"/kaggle/working/**/{gid}.py",
            f"**/game_sources/**/{gid}.py",
        ]
        for pattern in patterns:
            attempted.append(pattern)
            matches = glob.glob(pattern, recursive=True)
            if matches:
                src = matches[0]
                content = open(src).read()[:2000]
                m = re.search(r'class\s+(\w+)\s*\(\s*ARCBaseGame', content)
                if m:
                    cls_name = m.group(1)
                break

    if not src:
        logger.warning(
            f"BFS: game source not found for {game_id}. Paths attempted: {attempted}"
        )
    else:
        logger.info(f"BFS: found game source at {src} (paths tried before success not logged)")

    return src, cls_name


# =====================================================================
# SECTION: ForgeHypothesisAgent — the three-tier agent
# =====================================================================

class ForgeHypothesisAgent(Agent):
    """
    BFS-First Hybrid Agent for ARC-AGI-3 (Planning Document v4.0).

    Tier 1: Offline BFS solver (FORGE v19, bug-fixed)
    Tier 2: Pure-Python rule-discovery fallback (arc_prize_2026 utils, no LLM)
    Tier 3: FORGE v19 emergency heuristic

    No LLM, no model inference, no GPU required.
    """

    MAX_ACTIONS = 100  # Section 5.1

    def __init__(self, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)
        seed = int(time.time() * 1e6) + (hash(self.game_id) % 1000000)
        random.seed(seed)
        np.random.seed(seed % (2 ** 32 - 1))

        # ── Time budget (Section 4.4 — fixes the dead F2 code) ──────────────
        self.total_time_budget = 11 * 3600  # 11h of 12h session, 1h safety margin
        self.start_time = time.time()
        self.per_level_budget = 600
        self.levels_seen = 0
        self.level_start_time: Optional[float] = None
        self.estimated_total_levels = 50

        # ── Tier 1: BFS state ────────────────────────────────────────────────
        self._bfs: Optional[BFSSolver] = None
        self._bfs_solution: Optional[list] = None
        self._bfs_step: int = 0
        self._bfs_tried: bool = False
        self._bfs_failed_levels: Set[int] = set()
        self._current_level: int = -1

        # ── Tier 2: Rule-discovery state ────────────────────────────────────
        self.rule_book = RuleBook()
        self.state_graph = StateGraph()
        self.phase = Phase.EXPLORE
        self._prev_grid: Optional[list] = None
        self._prev_full_hash: Optional[str] = None
        self._prev_abstract_hash: Optional[str] = None
        self._last_action_name: Optional[str] = None
        self._prev_levels_completed: int = 0
        self._grid_history: List[list] = []
        self._player_delta_history: List[Any] = []
        self._level_changed_history: List[bool] = []
        self._state_changed_history: List[bool] = []
        self._abstract_hash_history: List[str] = []

    # ────────────────────────────────────────────────────────────────────
    # Agent interface
    # ────────────────────────────────────────────────────────────────────

    def is_done(self, frames: List[FrameData], latest_frame: FrameData) -> bool:
        try:
            return (
                latest_frame.state == GameState.WIN
                or (time.time() - self.start_time) >= self.total_time_budget
            )
        except Exception:
            return True

    def choose_action(self, frames: List[FrameData], latest_frame: FrameData) -> GameAction:
        try:
            return self._choose_action_impl(frames, latest_frame)
        except Exception as e:
            traceback.print_exc()
            return self._emergency_heuristic_safe(latest_frame, reason=str(e))

    # ────────────────────────────────────────────────────────────────────
    # Core decision loop (Section 5.2)
    # ────────────────────────────────────────────────────────────────────

    def _choose_action_impl(self, frames: List[FrameData], latest_frame: FrameData) -> GameAction:
        lvl = latest_frame.levels_completed

        if latest_frame.state in [GameState.NOT_PLAYED, GameState.GAME_OVER]:
            return self._make_action('RESET')

        if latest_frame.full_reset:
            self._reset_for_new_game()
            return self._make_action('RESET')

        if lvl != self._current_level:
            self._on_level_change(lvl)

        current_grid = latest_frame.frame[-1] if latest_frame.frame else None
        if current_grid is None:
            return self._make_action('ACTION1')

        self._sync_available_actions(latest_frame)

        # ── TIER 1: BFS EXECUTION ──────────────────────────────────────────
        if self._bfs_solution and self._bfs_step < len(self._bfs_solution):
            return self._execute_bfs_step()

        # ── TIER 1: BFS SOLVE ATTEMPT (only once per level) ─────────────────
        if not self._bfs_tried and lvl not in self._bfs_failed_levels:
            self._bfs_tried = True
            sol = self._try_bfs_solve(lvl)
            if sol:
                return self._execute_bfs_step()
            else:
                self._bfs_failed_levels.add(lvl)
                # fall through to Tier 2

        # ── TIER 2: RULE-DISCOVERY ───────────────────────────────────────────
        current_full_hash = get_full_hash(current_grid)
        current_abstract_hash = get_abstract_hash(current_grid, self.rule_book.decorative_values)

        # Transition cache fast path (Section 2.10)
        if self.rule_book.cached_bfs_path and self._prev_full_hash:
            next_step = self.rule_book.cached_bfs_path[0]
            cache_key = (self._prev_full_hash, next_step)
            if cache_key in self.rule_book.transition_cache:
                self.rule_book.cached_bfs_path.pop(0)
                self._bookkeeping(current_full_hash, current_abstract_hash, next_step)
                return self._make_action(next_step)

        self._grid_history.append(current_grid)
        if len(self._grid_history) > 20:
            self._grid_history.pop(0)
        self.rule_book.goal_candidates = detect_goal_candidates(
            self._grid_history, self.rule_book.impassable_values,
        )
        for row in current_grid:
            self.rule_book.known_values.update(row)

        self.phase = PhaseController.determine_phase(
            self.rule_book, self.action_counter, self.state_graph, current_abstract_hash
        )

        # Phase 1 fast path: systematic probe
        if self.phase == Phase.EXPLORE:
            next_probe = PhaseController.suggest_exploration_action(self.rule_book)
            if next_probe:
                self._bookkeeping(current_full_hash, current_abstract_hash, next_probe)
                return self._make_action(next_probe)

        # Phase 3 fast path: BFS step from rule-discovery path cache
        if self.phase == Phase.EXPLOIT and self.rule_book.cached_bfs_path:
            next_step = self.rule_book.cached_bfs_path.pop(0)
            self._bookkeeping(current_full_hash, current_abstract_hash, next_step)
            return self._make_action(next_step)

        # Full Tier 2 path: observe -> update rules -> select action
        delta = None
        if self._prev_grid is not None:
            delta = compute_delta(
                self._prev_grid, current_grid,
                self._prev_levels_completed, latest_frame.levels_completed,
                '', '',
                self.rule_book.known_values, self.rule_book.decorative_values,
            )
            self._player_delta_history.append(delta.player_delta)
            self._level_changed_history.append(delta.level_changed)
            self._state_changed_history.append(delta.state_changed)
            self._deterministic_rule_update(delta, self.rule_book)

            if self.action_counter > 10 and self.action_counter % 5 == 0:
                goal_vals = {v for v, _ in self.rule_book.goal_candidates}
                new_decorative = classify_decorative_values(
                    self._grid_history[-15:],
                    self._player_delta_history[-15:],
                    self._level_changed_history[-15:],
                    self._state_changed_history[-15:],
                    goal_vals,
                )
                self.rule_book.decorative_values.update(new_decorative)

        self._prev_levels_completed = latest_frame.levels_completed

        # Recompute BFS path if exploit phase and player/goal known
        if self.phase == Phase.EXPLOIT and not self.rule_book.cached_bfs_path:
            self._try_compute_bfs_path(current_grid)
            if self.rule_book.cached_bfs_path:
                next_step = self.rule_book.cached_bfs_path.pop(0)
                self._bookkeeping(current_full_hash, current_abstract_hash, next_step)
                return self._make_action(next_step)

        # Detect stuck -> recovery
        self._abstract_hash_history.append(current_abstract_hash)
        if self.state_graph.is_stuck(self._abstract_hash_history):
            self.phase = Phase.RECOVERY
            self.rule_book.cached_bfs_path = None

        self.state_graph.add_state(
            current_abstract_hash,
            action_taken=self._last_action_name,
            parent_hash=self._prev_abstract_hash,
        )

        # ── TIER 3: EMERGENCY HEURISTIC near action budget exhaustion ────────
        if self.action_counter > self.MAX_ACTIONS - 10:
            return self._emergency_heuristic(current_grid, latest_frame)

        action_name, x, y = self._deterministic_action_select(
            current_grid, self.rule_book, self.state_graph, current_abstract_hash, self.phase
        )

        self._bookkeeping(current_full_hash, current_abstract_hash, action_name)
        self._update_prev_state(current_grid, current_full_hash, current_abstract_hash, action_name)
        return self._make_action(action_name, x=x, y=y)

    # ────────────────────────────────────────────────────────────────────
    # Tier 2: deterministic rule update (Section 2.2 — replaces LLM Call 1)
    # ────────────────────────────────────────────────────────────────────

    def _deterministic_rule_update(self, delta: FrameDelta, rule_book: RuleBook) -> None:
        if self._last_action_name in ('ACTION1', 'ACTION2', 'ACTION3', 'ACTION4'):
            if delta.cells_changed == 0:
                if self._last_action_name not in rule_book.no_op_actions:
                    rule_book.no_op_actions.append(self._last_action_name)
            elif delta.player_delta is not None and delta.player_delta != (0, 0):
                direction = _ACTION_DIRECTIONS[self._last_action_name]
                if self._last_action_name not in rule_book.action_map:
                    rule_book.action_map[self._last_action_name] = ActionEffect(self._last_action_name)
                rule_book.action_map[self._last_action_name].effect_type = 'movement'
                rule_book.action_map[self._last_action_name].direction = direction
                rule_book.action_map[self._last_action_name].observation_count += 1
                rule_text = f'{self._last_action_name} moves player {direction}'
                if rule_text not in rule_book.confirmed_rules:
                    rule_book.confirmed_rules.append(rule_text)
            elif delta.player_delta == (0, 0) and delta.cells_changed <= 3:
                dr, dc = _DIRECTION_DELTAS.get(_ACTION_DIRECTIONS[self._last_action_name], (0, 0))
                if rule_book.player_position and self._prev_grid is not None:
                    r, c = rule_book.player_position
                    tr, tc = r + dr, c + dc
                    if 0 <= tr < len(self._prev_grid) and 0 <= tc < len(self._prev_grid[0]):
                        target_val = self._prev_grid[tr][tc]
                        if target_val not in rule_book.impassable_values:
                            rule_book.impassable_values.append(target_val)

        if self._last_action_name in rule_book.action_map:
            rule_book.action_map[self._last_action_name].observation_count += 1

        if delta.level_changed:
            if 'Level completed' not in rule_book.confirmed_rules:
                rule_book.confirmed_rules.append('Level completed')
            rule_book.levels_completed += 1
            if self._last_action_name:
                rule_book.win_condition = f'Take {self._last_action_name} near goal tile'

        if delta.new_values_seen:
            rule_book.known_values.update(delta.new_values_seen)

        if delta.player_delta and rule_book.player_position:
            r, c = rule_book.player_position
            dr, dc = delta.player_delta
            rule_book.player_position = (r + dr, c + dc)
        elif rule_book.player_position is None and delta.player_delta:
            # bootstrap: we don't know absolute position yet, can't infer from delta alone
            pass

    # ────────────────────────────────────────────────────────────────────
    # Tier 2: deterministic action selection (Section 2.3 — replaces LLM Call 2)
    # ────────────────────────────────────────────────────────────────────

    def _deterministic_action_select(
        self, current_grid, rule_book: RuleBook, state_graph: StateGraph,
        current_abstract_hash: str, phase: str,
    ) -> Tuple[str, int, int]:
        available = rule_book.available_actions or _ALL_ACTIONS

        if phase == Phase.EXPLORE:
            for action in available:
                if action not in rule_book.action_map:
                    return action, 0, 0

        if phase == Phase.EXPLOIT and rule_book.cached_bfs_path:
            return rule_book.cached_bfs_path.pop(0), 0, 0

        if (rule_book.win_condition and rule_book.player_position
                and rule_book.goal_candidates):
            goal_val = rule_book.goal_candidates[0][0]
            path = bfs_path(
                current_grid, rule_book.player_position, goal_val,
                rule_book.impassable_values,
            )
            if path:
                rule_book.cached_bfs_path = path
                if rule_book.cached_bfs_path:
                    return rule_book.cached_bfs_path.pop(0), 0, 0

        untried = state_graph.get_untried_actions(current_abstract_hash, available)
        if untried:
            for preferred in ('ACTION1', 'ACTION2', 'ACTION3', 'ACTION4', 'ACTION5'):
                if preferred in untried:
                    return preferred, 0, 0
            return untried[0], 0, 0

        backtrack = state_graph.suggest_backtrack(current_abstract_hash, available)
        if backtrack:
            return backtrack[0], 0, 0

        if 'ACTION6' in available and rule_book.goal_candidates:
            goal_val = rule_book.goal_candidates[0][0]
            pos = find_nearest_value(
                current_grid, rule_book.player_position or (32, 32), goal_val
            )
            if pos:
                r, c = pos
                return 'ACTION6', c, r  # x=col, y=row

        if available:
            return min(
                available,
                key=lambda a: rule_book.action_map.get(a, ActionEffect(a)).observation_count,
            ), 0, 0
        return 'ACTION1', 0, 0

    def _try_compute_bfs_path(self, current_grid) -> None:
        rb = self.rule_book
        if rb.win_condition and rb.player_position and rb.goal_candidates:
            goal_val = rb.goal_candidates[0][0]
            path = bfs_path(current_grid, rb.player_position, goal_val, rb.impassable_values)
            if path:
                rb.cached_bfs_path = path
                rb.cached_bfs_goal = (0, 0)

    # ────────────────────────────────────────────────────────────────────
    # Tier 1: BFS plumbing
    # ────────────────────────────────────────────────────────────────────

    def _init_bfs(self) -> None:
        src, cls = find_game_source_and_class(self.game_id, getattr(self, 'arc_env', None))
        if src:
            self._bfs = BFSSolver(src, cls, scan_timeout=5, bfs_timeout=180)
            if self._bfs.load():
                logger.info(f"BFS: loaded {cls} from {src}")
            else:
                self._bfs = None
                logger.warning("BFS: failed to load game class")
        else:
            self._bfs = None
            logger.warning(f"BFS: game source not found for {self.game_id}")

    def _try_bfs_solve(self, level_idx: int):
        """F1 fix: explicit timeout is computed and passed in — no bare
        undefined `bfs_timeout` name reference (was a NameError in FORGE v19)."""
        if self._bfs is None:
            return None
        prev_sol = self._bfs.solutions.get(level_idx - 1) if level_idx > 0 else None

        # Section 4.4: per-level time budget governs the BFS timeout
        bfs_timeout = min(120, max(20, self.per_level_budget * 0.6))

        sol = self._bfs.solve_level(level_idx, prev_solution=prev_sol, timeout=bfs_timeout)
        if sol:
            self._bfs_solution = sol
            self._bfs_step = 0
            return sol
        return None

    def _execute_bfs_step(self) -> GameAction:
        act_id, data = self._bfs_solution[self._bfs_step]
        self._bfs_step += 1
        action = GameAction.from_id(act_id)
        if data:
            action.set_data(data)
        action.reasoning = f"bfs:{self._bfs_step}/{len(self._bfs_solution)}"
        return action

    # ────────────────────────────────────────────────────────────────────
    # Tier 3: emergency heuristic (FORGE v19 _heuristic, ported)
    # ────────────────────────────────────────────────────────────────────

    def _emergency_heuristic(self, current_grid, latest_frame: FrameData) -> GameAction:
        frame = np.array(current_grid)
        avail_raw = getattr(latest_frame, 'available_actions', None) or []
        av = set(int(a.value) if hasattr(a, 'value') else int(a) for a in avail_raw)
        step = self.action_counter

        bg = int(np.bincount(frame.flatten(), minlength=16).argmax())

        for d in [1, 2, 3, 4]:
            if d in av and step % 4 < 2:
                action = GameAction.from_id(d)
                action.reasoning = 'tier3:directional-probe'
                return action

        if 6 in av:
            cnt = np.bincount(frame.flatten(), minlength=16)
            targets = []
            for c in range(16):
                if c == bg or cnt[c] == 0 or cnt[c] > 2000:
                    continue
                ys, xs = np.where(frame == c)
                if len(ys) >= 2:
                    targets.append((int(np.median(xs)), int(np.median(ys)), len(ys)))
            targets.sort(key=lambda t: t[2])
            if targets:
                x, y, _ = targets[0]
                action = GameAction.ACTION6
                action.set_data({'x': x, 'y': y})
                action.reasoning = 'tier3:rare-cell-click'
                return action

        if 5 in av:
            action = GameAction.ACTION5
            action.reasoning = 'tier3:interact'
            return action

        choices = [a for a in av if 1 <= a <= 5]
        if choices:
            action = GameAction.from_id(random.choice(choices))
            action.reasoning = 'tier3:random'
            return action

        action = GameAction.ACTION1
        action.reasoning = 'tier3:fallback'
        return action

    def _emergency_heuristic_safe(self, latest_frame: FrameData, reason: str = '') -> GameAction:
        """Used by the outer try/except in choose_action — never raises."""
        try:
            current_grid = latest_frame.frame[-1] if latest_frame.frame else [[0] * 64 for _ in range(64)]
            action = self._emergency_heuristic(current_grid, latest_frame)
            action.reasoning = f'err-fallback:{reason}'
            return action
        except Exception:
            action = GameAction.RESET
            action.reasoning = f'err-fallback-reset:{reason}'
            return action

    # ────────────────────────────────────────────────────────────────────
    # Bookkeeping / lifecycle
    # ────────────────────────────────────────────────────────────────────

    def _make_action(self, action_name: str, reasoning: Any = '', x: int = 0, y: int = 0) -> GameAction:
        try:
            action = GameAction.from_name(action_name)
        except (ValueError, KeyError):
            logger.warning(f"Unknown action name: {action_name}, defaulting to ACTION1")
            action = GameAction.ACTION1

        if action_name == 'ACTION6':
            grid_x = max(0, min(int(x), 63))
            grid_y = max(0, min(int(y), 63))
            action.set_data({'x': grid_x, 'y': grid_y})
        action.reasoning = reasoning or action_name
        return action

    def _sync_available_actions(self, latest_frame: FrameData) -> None:
        if getattr(latest_frame, 'available_actions', None):
            names = []
            for a in latest_frame.available_actions:
                names.append(a.name if hasattr(a, 'name') else str(a))
            self.rule_book.available_actions = names

    def _bookkeeping(self, current_full_hash, current_abstract_hash, action_name) -> None:
        self.state_graph.add_state(
            current_abstract_hash, action_taken=action_name, parent_hash=self._prev_abstract_hash,
        )
        self._last_action_name = action_name
        self._prev_full_hash = current_full_hash
        self._prev_abstract_hash = current_abstract_hash

    def _update_prev_state(self, current_grid, current_full_hash, current_abstract_hash, action_name) -> None:
        self._prev_grid = current_grid
        self._prev_full_hash = current_full_hash
        self._prev_abstract_hash = current_abstract_hash
        self._last_action_name = action_name

    def _on_level_change(self, new_level: int) -> None:
        """Section 4.4: per-level time budget update + carry-forward rules."""
        elapsed = time.time() - self.start_time
        remaining = max(1.0, self.total_time_budget - elapsed)
        levels_remaining_estimate = max(1, self.estimated_total_levels - self.levels_seen)
        self.per_level_budget = min(600, remaining / levels_remaining_estimate)
        self.level_start_time = time.time()
        self.levels_seen += 1

        if self._current_level >= 0:
            self.rule_book.carry_forward_for_new_level()
            self.state_graph = StateGraph()
            self._abstract_hash_history = []
            self._prev_grid = None

        self._current_level = new_level
        # Reset BFS attempt state for the new level (do NOT re-try failed levels)
        self._bfs_solution = None
        self._bfs_step = 0
        self._bfs_tried = False

        if self._bfs is None and not self._bfs_tried:
            self._init_bfs()

    def _reset_for_new_game(self) -> None:
        self.rule_book = RuleBook()
        self.state_graph = StateGraph()
        self.phase = Phase.EXPLORE
        self._prev_grid = None
        self._prev_full_hash = None
        self._prev_abstract_hash = None
        self._last_action_name = None
        self._prev_levels_completed = 0
        self._grid_history = []
        self._player_delta_history = []
        self._level_changed_history = []
        self._state_changed_history = []
        self._abstract_hash_history = []
        self._bfs_solution = None
        self._bfs_step = 0
        self._bfs_tried = False
        self._bfs_failed_levels = set()
        self._current_level = -1


Writing /kaggle/working/ARC-AGI-3-Agents/agents/templates/my_agent.py


## STEP 3 — Write `agents/__init__.py`

**Bug fix:** the v3.0 notebook's `agents/__init__.py` set
`AVAILABLE_AGENTS = {"hypothesisagent": None, ...}` and relied on a lazy `get_agent()`
function instead. But `swarm.py`'s `Swarm.__init__` does
`self.agent_class = AVAILABLE_AGENTS[agent]` directly — it never calls `get_agent()`.
That means `AVAILABLE_AGENTS[agent]` would have resolved to `None`, and `Swarm.main()`
would have crashed the first time it tried `None(card_id=..., game_id=..., ...)`.

v4 fixes this by registering the real class directly in `AVAILABLE_AGENTS`, exactly like
the working FORGE v19 `__init__.py` did (`AVAILABLE_AGENTS = {"random": Random, "myagent": MyAgent}`).

In [3]:
init_code = '''from typing import Type
from dotenv import load_dotenv
load_dotenv()

from .agent import Agent, Playback
from .swarm import Swarm
from .templates.random_agent import Random
from .templates.my_agent import ForgeHypothesisAgent

AVAILABLE_AGENTS: dict[str, Type[Agent]] = {
    "random": Random,
    "forgehypothesis": ForgeHypothesisAgent,
}
'''
with open('/kaggle/working/ARC-AGI-3-Agents/agents/__init__.py', 'w') as f:
    f.write(init_code)
print('agents/__init__.py written.')

agents/__init__.py written.


In [4]:
import sys
for key in list(sys.modules.keys()):
    if "agents" in key:
        del sys.modules[key]

## STEP 4 — Write `.env` configuration

In [5]:
import os

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    env_code = """SCHEME=http
HOST=gateway
PORT=8001
ARC_API_KEY=test-key-123
ARC_BASE_URL=http://gateway:8001/
OPERATION_MODE=online
ENVIRONMENTS_DIR=
RECORDINGS_DIR=/kaggle/working/server_recording
"""
    with open('/kaggle/working/ARC-AGI-3-Agents/.env', 'w') as f:
        f.write(env_code)
    print('.env written for competition mode.')
else:
    print('Local mode: .env not written (use your own .env file).')

Local mode: .env not written (use your own .env file).


In [6]:
import os

# Check competition input files
comp_path = '/kaggle/input/competitions/arc-prize-2026-arc-agi-3/'
if os.path.exists(comp_path):
    print('Competition input contents:')
    for f in os.listdir(comp_path):
        full = os.path.join(comp_path, f)
        if os.path.isdir(full):
            print(f'  [dir] {f}/')
            for sub in os.listdir(full)[:5]:
                print(f'      {sub}')
        else:
            size = os.path.getsize(full)
            print(f'  [file] {f} ({size/1024/1024:.1f} MB)')
else:
    print(f'{comp_path} not found (expected outside the competition rerun environment).')

print()
print('Environment variables:')
for key in ['ARC_API_KEY', 'KAGGLE_IS_COMPETITION_RERUN']:
    print(f'  {key}: {os.getenv(key, "NOT SET")}')

Competition input contents:
  [dir] environment_files/
      sk48
      tn36
      m0r0
      bp35
      cn04
  [dir] ARC-AGI-3-Agents/
      tests
      .pre-commit-config.yaml
      pytest.ini
      LICENSE
      .gitignore
  [dir] arc_agi_3_wheels/
      typing_extensions-4.15.0-py3-none-any.whl
      packaging-26.1-py3-none-any.whl
      charset_normalizer-3.4.7-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl
      fonttools-4.62.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl
      pyparsing-3.3.2-py3-none-any.whl

Environment variables:
  ARC_API_KEY: NOT SET
  KAGGLE_IS_COMPETITION_RERUN: NOT SET


## STEP 5 — Smoke test (local mode only)

Validates the pure-Python logic in `my_agent.py` — `RuleBook`, `StateGraph`, `bfs_path`,
`detect_goal_candidates`, `compute_delta`/`get_full_hash`/`get_abstract_hash`, and the
full `ForgeHypothesisAgent` three-tier dispatch loop — without requiring the competition
gateway or a real game source file. This replaces the v3.0 notebook's STEP 16 smoke test,
adapted for the single-file v4 structure (Planning Doc Section 6, Phase 1, task 6).

In [7]:
import sys
import os

if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    print('=== Smoke test: ForgeHypothesisAgent v4 (no gateway needed) ===')

    sys.path.insert(0, '/kaggle/working/ARC-AGI-3-Agents')

    import importlib.util

    spec = importlib.util.spec_from_file_location(
        'my_agent',
        '/kaggle/working/ARC-AGI-3-Agents/agents/templates/my_agent.py'
    )

    # my_agent.py imports `from agents.agent import Agent` and `from arcengine import ...`.
    # Provide minimal stand-ins so the smoke test can run without the full competition
    # package tree or the real arcengine wheel.
    import types

    if 'agents' not in sys.modules:
        agents_pkg = types.ModuleType('agents')
        agents_pkg.__path__ = ['/kaggle/working/ARC-AGI-3-Agents/agents']
        sys.modules['agents'] = agents_pkg

    try:
        import agents.agent  # real package, if available
    except Exception:
        agent_mod = types.ModuleType('agents.agent')

        class _StubAgent:
            MAX_ACTIONS = 80
            action_counter = 0
            def __init__(self, *a, **kw):
                self.game_id = kw.get('game_id', 'test')
            def is_done(self, frames, latest_frame):
                raise NotImplementedError
            def choose_action(self, frames, latest_frame):
                raise NotImplementedError

        agent_mod.Agent = _StubAgent
        sys.modules['agents.agent'] = agent_mod

    try:
        import arcengine  # real wheel, if available
    except Exception:
        from enum import Enum
        arcengine_mod = types.ModuleType('arcengine')

        class _GameState(Enum):
            NOT_PLAYED = 'NOT_PLAYED'
            GAME_OVER = 'GAME_OVER'
            WIN = 'WIN'
            PLAYING = 'PLAYING'

        class _ActionMeta:
            def __init__(self, value, name):
                self.value = value
                self.name = name
                self.reasoning = None
            def set_data(self, d):
                self._data = d
            def is_complex(self):
                return self.value == 6

        class _GameAction:
            RESET = _ActionMeta(0, 'RESET')
            ACTION1 = _ActionMeta(1, 'ACTION1')
            ACTION2 = _ActionMeta(2, 'ACTION2')
            ACTION3 = _ActionMeta(3, 'ACTION3')
            ACTION4 = _ActionMeta(4, 'ACTION4')
            ACTION5 = _ActionMeta(5, 'ACTION5')
            ACTION6 = _ActionMeta(6, 'ACTION6')
            ACTION7 = _ActionMeta(7, 'ACTION7')

            @staticmethod
            def from_id(i):
                m = {0:'RESET',1:'ACTION1',2:'ACTION2',3:'ACTION3',4:'ACTION4',5:'ACTION5',6:'ACTION6',7:'ACTION7'}
                return getattr(_GameAction, m[i])

            @staticmethod
            def from_name(n):
                return getattr(_GameAction, n)

        class _FrameData:
            def __init__(self, frame=None, state=_GameState.PLAYING, levels_completed=0,
                         full_reset=False, available_actions=None):
                self.frame = frame or []
                self.state = state
                self.levels_completed = levels_completed
                self.full_reset = full_reset
                self.available_actions = available_actions or []

        class _ActionInput:
            def __init__(self, id=None, data=None):
                self.id = id
                self.data = data

        arcengine_mod.GameState = _GameState
        arcengine_mod.GameAction = _GameAction
        arcengine_mod.FrameData = _FrameData
        arcengine_mod.ActionInput = _ActionInput
        sys.modules['arcengine'] = arcengine_mod

    ma = importlib.util.module_from_spec(spec)
    sys.modules['my_agent'] = ma
    spec.loader.exec_module(ma)

    from arcengine import FrameData, GameState, GameAction

    # ── RuleBook ──────────────────────────────────────────────────────
    try:
        rb = ma.RuleBook()
        rb.confirmed_rules = ['ACTION1 moves player up', 'Value 5 is a wall']
        rb.win_condition = 'Reach the yellow tile'
        print(f'  OK RuleBook: confirmed_rules={rb.confirmed_rules}')
    except Exception as e:
        print(f'  FAIL RuleBook: {e}')

    # ── StateGraph ────────────────────────────────────────────────────
    try:
        sg = ma.StateGraph()
        sg.add_state('hash_A')
        sg.add_state('hash_B', 'ACTION1', 'hash_A')
        sg.add_state('hash_C', 'ACTION2', 'hash_A')
        untried = sg.get_untried_actions('hash_A', ['ACTION1', 'ACTION2', 'ACTION3', 'ACTION4'])
        print(f'  OK StateGraph untried from hash_A: {untried} (expected ACTION3+ACTION4)')
        print(f'  OK is_dead_end with only 2 available: {sg.is_dead_end("hash_A", ["ACTION1", "ACTION2"])}')
    except Exception as e:
        print(f'  FAIL StateGraph: {e}')

    # ── pathfinder ────────────────────────────────────────────────────
    try:
        test_grid = [[0]*8 for _ in range(8)]
        test_grid[0][7] = 9
        path = ma.bfs_path(test_grid, (0, 0), 9, [5])
        print(f'  OK bfs_path: {path} (expected 7x ACTION4)')
    except Exception as e:
        print(f'  FAIL pathfinder: {e}')

    # ── goal_detector ─────────────────────────────────────────────────
    try:
        g = [[0]*64 for _ in range(64)]
        g[0][0] = 11
        candidates = ma.detect_goal_candidates([g, g], [], [])
        print(f'  OK detect_goal_candidates: {candidates[:2]}')
    except Exception as e:
        print(f'  FAIL goal_detector: {e}')

    # ── frame_diff ────────────────────────────────────────────────────
    try:
        h1 = ma.get_full_hash([[0]*64 for _ in range(64)])
        h2 = ma.get_abstract_hash([[0]*64 for _ in range(64)], set())
        print(f'  OK full_hash == abstract_hash (no decorative): {h1 == h2}')

        grid_a = [[0]*64 for _ in range(64)]
        grid_b = [[0]*64 for _ in range(64)]
        grid_b[5][5] = 9
        delta = ma.compute_delta(grid_a, grid_b, 0, 0, 'PLAYING', 'PLAYING', set(), set())
        print(f'  OK compute_delta: {delta.cells_changed} cell(s) changed (expected 1)')
    except Exception as e:
        print(f'  FAIL frame_diff: {e}')

    # ── ForgeHypothesisAgent end-to-end dispatch (no real BFS source available) ──
    try:
        class _TestAgent(ma.ForgeHypothesisAgent):
            def _init_bfs(self):
                self._bfs = None  # no game source in this smoke test

        agent = _TestAgent(
            card_id='c1', game_id='ls20-test', agent_name='forgehypothesis',
            ROOT_URL='http://x', record=False, arc_env=None, tags=[],
        )

        f0 = FrameData(frame=[], state=GameState.NOT_PLAYED)
        a0 = agent.choose_action([f0], f0)
        print(f'  OK NOT_PLAYED -> {a0.name} (expected RESET)')

        grid = [[0]*64 for _ in range(64)]
        grid[10][10] = 9
        avail = [GameAction.ACTION1, GameAction.ACTION2, GameAction.ACTION3,
                 GameAction.ACTION4, GameAction.ACTION6]
        f1 = FrameData(frame=[grid], state=GameState.PLAYING, levels_completed=0,
                        available_actions=avail)

        names = []
        for i in range(20):
            agent.action_counter = i
            a = agent.choose_action([f1], f1)
            names.append(a.name)
        print(f'  OK 20-step dispatch loop, no crash. Actions: {names}')

        agent.action_counter = agent.MAX_ACTIONS - 5
        a_tier3 = agent.choose_action([f1], f1)
        print(f'  OK Tier-3 emergency heuristic near budget exhaustion: {a_tier3.name} ({a_tier3.reasoning})')

    except Exception as e:
        import traceback
        traceback.print_exc()
        print(f'  FAIL ForgeHypothesisAgent integration: {e}')

    print()
    print('=== Smoke test complete. Run STEP 6 on Kaggle competition to play. ===')
else:
    print('Competition mode: skipping smoke test.')

=== Smoke test: ForgeHypothesisAgent v4 (no gateway needed) ===
  OK RuleBook: confirmed_rules=['ACTION1 moves player up', 'Value 5 is a wall']
  OK StateGraph untried from hash_A: ['ACTION3', 'ACTION4'] (expected ACTION3+ACTION4)
  OK is_dead_end with only 2 available: True
  OK bfs_path: ['ACTION4', 'ACTION4', 'ACTION4', 'ACTION4', 'ACTION4', 'ACTION4', 'ACTION4'] (expected 7x ACTION4)
  OK detect_goal_candidates: [(11, 0.75)]
  OK full_hash == abstract_hash (no decorative): True
  OK compute_delta: 1 cell(s) changed (expected 1)
  OK NOT_PLAYED -> RESET (expected RESET)
  OK 20-step dispatch loop, no crash. Actions: ['ACTION1', 'ACTION1', 'ACTION1', 'ACTION1', 'ACTION1', 'ACTION1', 'ACTION2', 'ACTION3', 'ACTION4', 'ACTION6', 'ACTION6', 'ACTION6', 'ACTION6', 'ACTION6', 'ACTION6', 'ACTION6', 'ACTION6', 'ACTION6', 'ACTION6', 'ACTION6']
  OK Tier-3 emergency heuristic near budget exhaustion: ACTION1 (tier3:random)

=== Smoke test complete. Run STEP 6 on Kaggle competition to play. ===


## STEP 6 — Write `main.py`

In [8]:
%%writefile /kaggle/working/ARC-AGI-3-Agents/main.py
# ruff: noqa: E402
from dotenv import load_dotenv

load_dotenv(dotenv_path=".env.example")
load_dotenv(dotenv_path=".env", override=True)

import argparse
import json
import logging
import os
import signal
import sys
import threading
from functools import partial
from types import FrameType
from typing import Optional

import requests

from agents import AVAILABLE_AGENTS, Swarm
from agents.tracing import initialize as init_agentops

logger = logging.getLogger()

SCHEME = os.environ.get("SCHEME", "http")
HOST = os.environ.get("HOST", "localhost")
PORT = os.environ.get("PORT", 8001)

# Hide standard ports in URL
if (SCHEME == "http" and str(PORT) == "80") or (
    SCHEME == "https" and str(PORT) == "443"
):
    ROOT_URL = f"{SCHEME}://{HOST}"
else:
    ROOT_URL = f"{SCHEME}://{HOST}:{PORT}"
HEADERS = {
    "X-API-Key": os.getenv("ARC_API_KEY", ""),
    "Accept": "application/json",
}


def run_agent(swarm: Swarm) -> None:
    swarm.main()
    os.kill(os.getpid(), signal.SIGINT)


def cleanup(
    swarm: Swarm,
    signum: Optional[int],
    frame: Optional[FrameType],
) -> None:
    logger.info("Received SIGINT, exiting...")
    card_id = swarm.card_id
    if card_id:
        scorecard = swarm.close_scorecard(card_id)
        if scorecard:
            logger.info("--- EXISTING SCORECARD REPORT ---")
            logger.info(json.dumps(scorecard.model_dump(), indent=2))
            swarm.cleanup(scorecard)

        # Provide web link to scorecard
        if card_id:
            scorecard_url = f"{ROOT_URL}/scorecards/{card_id}"
            logger.info(f"View your scorecard online: {scorecard_url}")

    sys.exit(0)


def main() -> None:
    log_level = logging.INFO
    if os.environ.get("DEBUG", "False") == "True":
        log_level = logging.DEBUG

    logger.setLevel(log_level)
    formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")

    stdout_handler = logging.StreamHandler(sys.stdout)
    stdout_handler.setLevel(log_level)
    stdout_handler.setFormatter(formatter)

    file_handler = logging.FileHandler("logs.log", mode="w")
    file_handler.setLevel(log_level)
    file_handler.setFormatter(formatter)

    logger.addHandler(file_handler)
    logger.addHandler(stdout_handler)

    # logging.getLogger("requests").setLevel(logging.CRITICAL)
    # logging.getLogger("werkzeug").setLevel(logging.CRITICAL)

    parser = argparse.ArgumentParser(description="ARC-AGI-3-Agents")
    parser.add_argument(
        "-a",
        "--agent",
        choices=AVAILABLE_AGENTS.keys(),
        help="Choose which agent to run.",
    )
    parser.add_argument(
        "-g",
        "--game",
        help="Choose a specific game_id for the agent to play. If none specified, an agent swarm will play all available games.",
    )
    parser.add_argument(
        "-t",
        "--tags",
        type=str,
        help="Comma-separated list of tags for the scorecard (e.g., 'experiment,v1.0')",
        default=None,
    )

    args = parser.parse_args()

    if not args.agent:
        logger.error("An Agent must be specified")
        return

    print(f"{ROOT_URL}/api/games")

    # Get the list of games from the API
    full_games = []
    try:
        with requests.Session() as session:
            session.headers.update(HEADERS)
            r = session.get(f"{ROOT_URL}/api/games", timeout=10)

        if r.status_code == 200:
            try:
                full_games = [g["game_id"] for g in r.json()]
            except (ValueError, KeyError) as e:
                logger.error(f"Failed to parse games response: {e}")
                logger.error(f"Response content: {r.text[:200]}")
        else:
            logger.error(
                f"API request failed with status {r.status_code}: {r.text[:200]}"
            )

    except requests.exceptions.RequestException as e:
        logger.error(f"Failed to connect to API server: {e}")

    # For playback agents, we can derive the game from the recording filename
    if not full_games and args.agent and args.agent.endswith(".recording.jsonl"):
        from agents.recorder import Recorder

        game_prefix = Recorder.get_prefix_one(args.agent)
        full_games = [game_prefix]
        logger.info(
            f"Using game '{game_prefix}' derived from playback recording filename"
        )
    games = full_games[:]
    if args.game:
        filters = args.game.split(",")
        games = [
            gid
            for gid in full_games
            if any(gid.startswith(prefix) for prefix in filters)
        ]

    logger.info(f"Game list: {games}")

    if not games:
        if full_games:
            logger.error(
                f"The specified game '{args.game}' does not exist or is not available with your API key. Please try a different game."
            )
        else:
            logger.error(
                "No games available to play. Check API connection or recording file."
            )
        return

    # Start with Empty tags, "agent" and agent name will be added by the Swarm later
    tags: list[str] = []

    # Append user-provided tags if any
    if args.tags:
        user_tags = [tag.strip() for tag in args.tags.split(",")]
        tags.extend(user_tags)

    # Initialize AgentOps client
    init_agentops(api_key=os.getenv("AGENTOPS_API_KEY"), log_level=log_level)

    swarm = Swarm(
        args.agent,
        ROOT_URL,
        games,
        tags=tags,  # Pass tags as keyword argument
    )
    agent_thread = threading.Thread(target=partial(run_agent, swarm))
    agent_thread.daemon = True  # die when the main thread dies
    agent_thread.start()

    signal.signal(signal.SIGINT, partial(cleanup, swarm))  # handler for Ctrl+C

    try:
        # Wait for the agent thread to complete
        while agent_thread.is_alive():
            agent_thread.join(timeout=5)  # Check every 5 second
    except KeyboardInterrupt:
        logger.info("KeyboardInterrupt received in main thread")
        cleanup(swarm, signal.SIGINT, None)
    except Exception as e:
        logger.error(f"Unexpected error in main thread: {e}")
        cleanup(swarm, None, None)


if __name__ == "__main__":
    os.environ["TESTING"] = "False"
    main()


Overwriting /kaggle/working/ARC-AGI-3-Agents/main.py


## STEP 7 — Run ForgeHypothesisAgent (competition) or dummy submission (local)

In [9]:
import os
import sys
import subprocess
import pandas as pd

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    print('=== Running ForgeHypothesisAgent (BFS-First Hybrid, v4.0) on competition games ===')
    print('Agent: forgehypothesis')
    print('MAX_ACTIONS: 100')
    print('Tiers: 1) Offline BFS solver  2) Pure-Python rule-discovery  3) Emergency heuristic')
    print('No LLM calls. No GPU required.')
    print()

    main_path = '/kaggle/working/ARC-AGI-3-Agents/main.py'
    if not os.path.exists(main_path):
        raise FileNotFoundError('main.py not found - make sure the %%writefile cell above ran')

    result = subprocess.run(
        [sys.executable, 'main.py', '-a', 'forgehypothesis'],
        cwd='/kaggle/working/ARC-AGI-3-Agents',
        env={**os.environ, 'MPLBACKEND': 'agg'},
    )
    print(f'\nmain.py exit code: {result.returncode}')

    sub_path = '/kaggle/working/submission.parquet'
    if os.path.exists(sub_path):
        df = pd.read_parquet(sub_path)
        print(f'Submission rows: {len(df)}')
        print(df.head())
    else:
        raise FileNotFoundError('submission.parquet was NOT written - check logs above')

else:
    print('Local mode: creating dummy submission parquet.')
    submission = pd.DataFrame(
        data=[['1_0', '1', True, 1]],
        columns=['row_id', 'game_id', 'end_of_game', 'score']
    )
    submission.to_parquet('/kaggle/working/submission.parquet', index=False)
    print(submission.head())
    print()
    print('Ready. Set KAGGLE_IS_COMPETITION_RERUN=1 and deploy to Kaggle.')

Local mode: creating dummy submission parquet.
  row_id game_id  end_of_game  score
0    1_0       1         True      1

Ready. Set KAGGLE_IS_COMPETITION_RERUN=1 and deploy to Kaggle.


---

## Architecture Summary

| Component | Location in `my_agent.py` | Planning Doc Section |
|---|---|---|
| Frame delta + two-tier hashing | `FrameDelta`, `compute_delta`, `get_full_hash`, `get_abstract_hash` | 2.3, 2.11 |
| Persistent rule accumulator | `RuleBook`, `ActionEffect`, `ObjectDef` | 2.4 |
| Multi-heuristic goal detection | `detect_goal_candidates` | 2.6 |
| BFS pathfinder (rule-discovery tier) | `bfs_path`, `find_nearest_value` | 2.7 |
| Stuck-state prevention | `StateGraph`, `StateNode` | 2.5, 2.11 |
| Phase policy (no LLM effort policy) | `Phase`, `PhaseController` | 2.8 |
| Tier 1 — offline BFS solver | `BFSSolver`, `find_game_source_and_class` | 1.1, 4.1, 4.3 |
| Tier 2 — deterministic rule update / action select | `ForgeHypothesisAgent._deterministic_rule_update`, `._deterministic_action_select` | 2.2, 2.3 |
| Tier 3 — emergency heuristic | `ForgeHypothesisAgent._emergency_heuristic` | 5.2 |
| Time budget per level | `ForgeHypothesisAgent._on_level_change` | 4.4 |

**What changed vs the v3.0 (Qwen2-VL) notebook:**
- Zero LLM calls anywhere — Tiers 1–3 are all deterministic/algorithmic.
- No `autoawq`, `qwen-vl-utils`, or any model weights to download.
- `agents/utils/*` package collapsed into one file per Section 7 ("single-file" rule).
- `prompt_builder.py`, the Qwen2-VL `hypothesis_agent.py`, and `reasoning_agent_fixed.py`
  are dropped entirely (Section 1.3 / Section 9, item 2).
- `agents/__init__.py`'s `AVAILABLE_AGENTS` bug (mapped to `None` instead of the class)
  is fixed so `Swarm` can actually instantiate the agent.
- All FORGE v19 bugs F1–F7 are fixed in `BFSSolver`/`find_game_source_and_class`.

**Success metrics targeted (Planning Doc Section 8):**
- Leaderboard score: 0.40 (FORGE v19 baseline) -> 0.55-0.65 target
- BFS success rate: >= 60% of levels attempted
- Tier 2 win rate on BFS-failed levels: >= 20%
- Zero BFS `NameError` crashes (F1 fixed)
- All 55 games complete within the 12-hour session